# MiniMax-H3 复测：默认只做预检

独立子进程 NCCL 已经在两张 Tesla T4 上通过，sum 为 5，两张卡 UUID 不同。那不是 Ray `H3T4Worker` 全链路，也不是视频生成。

本笔记本默认 `H3_T4_STAGE=preflight`。它只检查本机现在挂载的文件和磁盘，不下载权重，不启动 Ray，不替换 Kaggle 的 CUDA torch。

权重上次在已停止会话的临时目录里，不能假定还在。缺文件时结果是 `MISSING`。下载清单写在报告里，等你确认缺失后再决定要不要下，不要在这里自动下约 40GB。

`/kaggle/working` 上次大约只有 20GB，装不下全部官方权重。不要为了这次复测新建或发布权重 Dataset。

时间：没有权重文件时，Run All 应在几分钟内结束，只下载一个小 JSON。不要额外点安装或打开别的页面。只有把环境变量改成 `actor-admission` 才启动真实 actor，硬上限大约 6 分钟，然后先下载证据 zip。


In [ ]:
import os, sys
from pathlib import Path

def find_root():
    seeds = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    seen = []
    for seed in seeds:
        if seed.exists():
            seen.extend([seed, *seed.glob("*"), *seed.glob("*/*")])
    for candidate in seen:
        if (candidate / "scripts" / "model_retest.py").is_file():
            return candidate
    raise SystemExit("未找到 scripts/model_retest.py。上传本文件夹到 /kaggle/working。")

ROOT = find_root()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
STAGE = os.environ.get("H3_T4_STAGE", "preflight")
print({"root": str(ROOT), "stage": STAGE, "download": False})


In [ ]:
import json
from pathlib import Path
from scripts.model_retest import main
EVIDENCE = Path("/kaggle/working/h3-model-retest")
code = main(["--stage", "preflight", "--search", "/kaggle/input", "--search", "/kaggle/working", "--evidence-dir", str(EVIDENCE)])
report = json.loads((EVIDENCE / "preflight.json").read_text(encoding="utf-8"))
print(json.dumps({"exit": code, "inventory": report["inventory"]["status"], "missing": report["inventory"]["missing"], "inference_allowed": report["inference_allowed"], "download": report["download"]}, ensure_ascii=False))
if report["inventory"]["status"] == "MISSING":
    print("MISSING。报告里的 download_list_if_missing 只是清单，本笔记本不会下载。")


## Ray actor 准入

只有在你把 `H3_T4_STAGE` 设为 `actor-admission` 时才运行。它走原来的 `H3T4Initializer` / `H3T4Worker`，做 ready 和一次不读权重的 allreduce，然后关闭 actor。通过也不表示显存或内存够装模型。


In [ ]:
if STAGE != "actor-admission":
    print("跳过 actor 准入。默认预检不会启动 Ray。")
else:
    code = main(["--stage", "actor-admission", "--search", "/kaggle/input", "--search", "/kaggle/working", "--evidence-dir", str(EVIDENCE)])
    print({"actor_exit": code})
    print("先下载 /kaggle/working/h3-model-retest 里的 actor_admission.json、evidence.zip 和 sha256，再停止 session。")


## 模型加载

默认不运行。即便文件齐全且 actor 准入通过，这一步也只说明门禁结果，不启动 T2V/I2V，因为文件大小不能证明运行峰值放得下。


In [ ]:
if STAGE != "model-load":
    print("跳过模型加载。inference_allowed 保持 false。")
else:
    code = main(["--stage", "model-load", "--search", "/kaggle/input", "--search", "/kaggle/working", "--evidence-dir", str(EVIDENCE)])
    print({"model_exit": code})
